In [1]:
# ## Cell 1: Thiết lập thư viện và môi trường
# %%
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import copy
import time
from collections import defaultdict
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Thiết lập device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Thiết lập seed để reproduce kết quả
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)

torch.backends.cudnn.benchmark = True

Using device: cuda
GPU: Tesla T4
Memory: 14.7 GB


In [2]:
# ## Cell 2: Tải và chuẩn bị dữ liệu MNIST

transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Lambda(lambda x: x.view(-1))  # làm phẳng 28x28 -> 784
])

# Tải tập dữ liệu MNIST - SỬ DỤNG KAGGLE DATASETS
train_dataset = datasets.MNIST(root='/kaggle/working/mnist', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='/kaggle/working/mnist', train=False, download=True, transform=transform)

# Tạo DataLoaderđể tối ưu GPU
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Kích thước tập huấn luyện: {len(train_dataset)}")
print(f"Kích thước tập kiểm tra: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 39.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.10MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.82MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.02MB/s]

Kích thước tập huấn luyện: 60000
Kích thước tập kiểm tra: 10000


In [3]:
# ## Cell 3: Định nghĩa mô hình MLP

class MemristorMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()  # hàm kích hoạt sigmoid phù hợp với memristor
        
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x

model = MemristorMLP().to(device)
criterion = nn.CrossEntropyLoss()
epochs = 200


print(model)
print(f"Mô hình đang chạy trên: {next(model.parameters()).device}")

MemristorMLP(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (sigmoid): Sigmoid()
)
Mô hình đang chạy trên: cuda:0


In [4]:
# ## Cell 4: Huấn luyện mô hình lý tưởng
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

# early stopping
patience = 5
best_test_acc = 0.0
epochs_no_improve = 0
best_model_state = None

start_time = time.time()

for epoch in range(epochs):
    lr = 0.2 / (1 + 0.01*epochs);
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum = 0.4)
    # Huấn luyện
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        _, pred = out.max(1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    
    train_acc = correct / total * 100
    train_loss = total_loss / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Đánh giá
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            test_loss += loss.item() * x.size(0)
            _, pred = out.max(1)
            test_correct += (pred == y).sum().item()
            test_total += x.size(0)
    
    test_acc = test_correct / test_total * 100
    test_loss_val = test_loss / test_total
    test_losses.append(test_loss_val)
    test_accuracies.append(test_acc)

    # Early stopping logic
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        epochs_no_improve = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}/{epochs}, "
          f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, "
          f"Test Loss={test_loss_val:.4f}, Test Acc={test_acc:.2f}%, "
          f"No improve: {epochs_no_improve}/{patience}")

    # Check early stopping
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

Epoch 1/200, Train Loss=1.1314, Train Acc=69.21%, Test Loss=0.5133, Test Acc=87.25%, No improve: 0/5
Epoch 2/200, Train Loss=0.4494, Train Acc=87.90%, Test Loss=0.3770, Test Acc=89.57%, No improve: 0/5
Epoch 3/200, Train Loss=0.3707, Train Acc=89.54%, Test Loss=0.3355, Test Acc=90.47%, No improve: 0/5
Epoch 4/200, Train Loss=0.3409, Train Acc=90.17%, Test Loss=0.3153, Test Acc=90.91%, No improve: 0/5
Epoch 5/200, Train Loss=0.3232, Train Acc=90.72%, Test Loss=0.3045, Test Acc=91.42%, No improve: 0/5
Epoch 6/200, Train Loss=0.3116, Train Acc=91.00%, Test Loss=0.2960, Test Acc=91.43%, No improve: 0/5
Epoch 7/200, Train Loss=0.3024, Train Acc=91.30%, Test Loss=0.2925, Test Acc=91.63%, No improve: 0/5
Epoch 8/200, Train Loss=0.2957, Train Acc=91.52%, Test Loss=0.2827, Test Acc=91.79%, No improve: 0/5
Epoch 9/200, Train Loss=0.2895, Train Acc=91.70%, Test Loss=0.2755, Test Acc=92.16%, No improve: 0/5
Epoch 10/200, Train Loss=0.2838, Train Acc=91.83%, Test Loss=0.2749, Test Acc=91.93%, No im

In [5]:
# ## Cell 4: Lưu trọng số mô hình lý tưởng
import copy
# Khôi phục mô hình tốt nhất
if best_model_state is not None:
    model.load_state_dict(best_model_state)

end_time = time.time()
print(f"Thời gian huấn luyện: {end_time - start_time:.2f} giây")

# Lưu trữ trạng thái mô hình lý tưởng
original_state_dict = copy.deepcopy(model.state_dict())

print(f"Best test accuracy: {best_test_acc:.2f}%")
import scipy.io as sio

# Lưu file .mat
W1_ideal = model.fc1.weight.detach().cpu().numpy()
b1_ideal = model.fc1.bias.detach().cpu().numpy()
W2_ideal = model.fc2.weight.detach().cpu().numpy()
b2_ideal = model.fc2.bias.detach().cpu().numpy()

sio.savemat("mlp_weights-BestAdam.mat", {"W1": W1_ideal, "b1": b1_ideal, "W2": W2_ideal, "b2": b2_ideal})
print("Đã lưu trọng số LÝ TƯỞNG")


Thời gian huấn luyện: 428.07 giây
Best test accuracy: 97.25%
Đã lưu trọng số LÝ TƯỞNG


In [6]:
# ## Cell 6: Mô phỏng nhiễu memristor với nhiều mức độ (ABSOLUTE NOISE)

# Danh sách các mức nhiễu cần test 
noise_levels = [0.05, 0.1, 0.15, 0.2, 0.30, 0.50]
results = []


for noise_level in noise_levels:
    # Khôi phục mô hình về baseline trước khi thêm nhiễu
    model.load_state_dict(original_state_dict)
    
    print(f"\n--- Testing với ABSOLUTE noise level: {noise_level:.2f} ---")
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' in name:
                noise = torch.randn_like(param) * noise_level
                param.add_(noise) 
    
    # Đánh giá sau khi thêm nhiễu
    model.eval()
    correct = 0
    total = 0
    test_loss = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            test_loss += loss.item() * x.size(0)
            _, pred = out.max(1)
            correct += (pred == y).sum().item()
            total += x.size(0)

    acc_after_noise = correct / total * 100
    avg_loss = test_loss / total
    
    # Tính độ suy giảm so với baseline
    baseline_acc = 97.33  # Từ kết quả trước của bạn
    accuracy_drop = baseline_acc - acc_after_noise
    
    results.append({
        'noise_level': noise_level,
        'accuracy': acc_after_noise,
        'loss': avg_loss,
        'accuracy_drop': accuracy_drop
    })
    
    print(f"Absolute Noise Level: {noise_level:.2f}")
    print(f"Test Accuracy = {acc_after_noise:.2f}% (Drop: {accuracy_drop:.2f}%)")
    print(f"Test Loss = {avg_loss:.4f}")

# Hiển thị kết quả tổng hợp
print("\n" + "="*70)
print("TỔNG HỢP KẾT QUẢ VỚI NHIỄU TUYỆT ĐỐI")
print("="*70)
print(f"{'Noise Level':<12} {'Accuracy':<12} {'Loss':<12} {'Accuracy Drop':<15}")
print("-" * 70)
print(f"{'Baseline':<12} {baseline_acc:<12.2f} {'-':<12} {'-':<15}")
for result in results:
    print(f"{result['noise_level']:<12.2f} {result['accuracy']:<12.2f} {result['loss']:<12.4f} {result['accuracy_drop']:<15.2f}")



--- Testing với ABSOLUTE noise level: 0.05 ---
Absolute Noise Level: 0.05
Test Accuracy = 96.66% (Drop: 0.67%)
Test Loss = 0.1053

--- Testing với ABSOLUTE noise level: 0.10 ---
Absolute Noise Level: 0.10
Test Accuracy = 94.64% (Drop: 2.69%)
Test Loss = 0.1659

--- Testing với ABSOLUTE noise level: 0.15 ---
Absolute Noise Level: 0.15
Test Accuracy = 91.59% (Drop: 5.74%)
Test Loss = 0.2593

--- Testing với ABSOLUTE noise level: 0.20 ---
Absolute Noise Level: 0.20
Test Accuracy = 69.26% (Drop: 28.07%)
Test Loss = 1.0733

--- Testing với ABSOLUTE noise level: 0.30 ---
Absolute Noise Level: 0.30
Test Accuracy = 34.58% (Drop: 62.75%)
Test Loss = 3.8776

--- Testing với ABSOLUTE noise level: 0.50 ---
Absolute Noise Level: 0.50
Test Accuracy = 23.13% (Drop: 74.20%)
Test Loss = 7.7202

TỔNG HỢP KẾT QUẢ VỚI NHIỄU TUYỆT ĐỐI
Noise Level  Accuracy     Loss         Accuracy Drop  
----------------------------------------------------------------------
Baseline     97.33        -            -       

In [7]:
## Cell 7: Huấn luyện với Random Weight Perturbation (RWP) 

patience = 10
best_noisy_acc = 0.0
epochs_no_improve = 0
max_epochs = 200
convergence_epoch = None

from_scratch = True

# Tạo model RWP với khởi tạo ngẫu nhiên
model_rwp = MemristorMLP().to(device)

initial_lr = 0.2 
optimizer_rwp = optim.SGD(model_rwp.parameters(), lr=initial_lr, momentum=0.4)

# Thiết lập noise & loss
noise_std = 0.09
alpha = 0.3
num_noise_samples = 3
combine_loss = True

# Lưu lịch sử
history = {
    'train_acc': [], 'train_loss': [],
    'test_clean_acc': [], 'test_noisy_acc': [],
    'learning_rates': []  # Theo dõi lr
}

start_time = time.time()

for epoch in range(max_epochs):
    current_lr = initial_lr / (1 + 0.01 * epoch)  
    for param_group in optimizer_rwp.param_groups:
        param_group['lr'] = current_lr
    
    model_rwp.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x = x.view(x.size(0), -1)
        optimizer_rwp.zero_grad()
        
        # --- Forward clean ---
        out_clean = model_rwp(x)
        loss_clean = criterion(out_clean, y)

        # --- Forward noisy với gradient ---
        loss_noisy = 0.0
        for _ in range(num_noise_samples):
            # Lưu weights gốc
            original_weights = {}
            with torch.no_grad():
                for name, param in model_rwp.named_parameters():
                    original_weights[name] = param.data.clone()
            
            # Thêm noise vào weights
            with torch.no_grad():
                for name, param in model_rwp.named_parameters():
                    if 'weight' in name:
                        noise = torch.randn_like(param) * noise_std
                        param.data.add_(noise)

            # Forward noisy (CÓ gradient)
            out_noisy = model_rwp(x)
            loss_noisy_sample = criterion(out_noisy, y)
            loss_noisy += loss_noisy_sample

            # Khôi phục weights gốc
            with torch.no_grad():
                for name, param in model_rwp.named_parameters():
                    if name in original_weights:
                        param.data.copy_(original_weights[name])
        
        loss_noisy /= num_noise_samples

        # --- Kết hợp loss ---
        if combine_loss:
            loss = alpha * loss_clean + (1.0 - alpha) * loss_noisy
        else:
            loss = loss_noisy

        # --- Backward và update ---
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rwp.parameters(), max_norm=1.0)
        optimizer_rwp.step()

        # Thống kê train acc/loss
        total_loss += loss.item() * x.size(0)
        _, pred = out_clean.max(1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    train_acc = correct / total * 100
    train_loss = total_loss / total
    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['learning_rates'].append(current_lr)  # Lưu lr mỗi epoch

    # ===== Evaluation =====
    model_rwp.eval()

    # --- Clean test ---
    clean_correct, clean_total = 0, 0
    with torch.no_grad():
        for x_t, y_t in test_loader:
            x_t, y_t = x_t.to(device), y_t.to(device)
            x_t = x_t.view(x_t.size(0), -1)
            out = model_rwp(x_t)
            _, pred = out.max(1)
            clean_correct += (pred == y_t).sum().item()
            clean_total += x_t.size(0)
    test_clean_acc = clean_correct / clean_total * 100
    history['test_clean_acc'].append(test_clean_acc)

    # --- Noisy test ---
    noisy_correct, noisy_total = 0, 0
    num_test_samples = 5
    
    for _ in range(num_test_samples):
        # Lưu weights gốc
        original_weights = {}
        with torch.no_grad():
            for name, param in model_rwp.named_parameters():
                if 'weight' in name:
                    original_weights[name] = param.data.clone()
        
        # Thêm noise
        with torch.no_grad():
            for name, param in model_rwp.named_parameters():
                if 'weight' in name:
                    noise = torch.randn_like(param) * noise_std
                    param.data.add_(noise)
        
        # Test với noisy weights
        with torch.no_grad():
            for x_t, y_t in test_loader:
                x_t, y_t = x_t.to(device), y_t.to(device)
                x_t = x_t.view(x_t.size(0), -1)
                out = model_rwp(x_t)
                _, pred = out.max(1)
                noisy_correct += (pred == y_t).sum().item()
                noisy_total += x_t.size(0)
        
        # Restore weights gốc
        with torch.no_grad():
            for name, param in model_rwp.named_parameters():
                if name in original_weights:
                    param.data.copy_(original_weights[name])
    
    test_noisy_acc = noisy_correct / noisy_total * 100
    history['test_noisy_acc'].append(test_noisy_acc)

    # --- Log ---
    print(f"Epoch {epoch+1}/{max_epochs}: "
          f"LR={current_lr:.4f}, TrainLoss={train_loss:.4f}, TrainAcc={train_acc:.2f}%, "
          f"TestClean={test_clean_acc:.2f}%, TestNoisy={test_noisy_acc:.2f}%")

    # Early stopping với adaptive learning rate
    current_score = 0.6 * test_noisy_acc + 0.4 * test_clean_acc
    
    if current_score > best_noisy_acc:
        best_noisy_acc = current_score
        epochs_no_improve = 0
        best_model_state = {k: v.clone() for k, v in model_rwp.state_dict().items()}
        convergence_epoch = epoch + 1
        best_actual_noisy = test_noisy_acc
        best_actual_clean = test_clean_acc
        print(f"  ↳ New best: Noisy={test_noisy_acc:.2f}%, Clean={test_clean_acc:.2f}%")
    else:
        epochs_no_improve += 1
        # Giảm LR nếu không cải thiện sau 1/2 patience
        if epochs_no_improve >= patience // 2:
            current_lr *= 0.95  # Giảm LR nhẹ
            for param_group in optimizer_rwp.param_groups:
                param_group['lr'] = current_lr
            print(f"  ↳ Reducing LR to {current_lr:.4f}")

    if epochs_no_improve >= patience:
        print(f"\nEarly stopping after {patience} epochs without improvement")
        break

# Khôi phục best model
model_rwp.load_state_dict(best_model_state)

end_time = time.time()
print(f"\nThời gian huấn luyện RWP: {end_time - start_time:.2f} giây")

print("\n=== KẾT QUẢ HUẤN LUYỆN ===")
print(f"Hội tụ tại epoch: {convergence_epoch}")
print(f"Best noisy accuracy: {best_actual_noisy:.2f}%")
print(f"Best clean accuracy: {best_actual_clean:.2f}%")
print(f"Final clean accuracy: {history['test_clean_acc'][-1]:.2f}%")

# Tính robustness gap
robustness_gap = best_actual_clean - best_actual_noisy
print(f"Robustness Gap: {robustness_gap:.2f}%")


# Lưu trọng số
print("Lưu trọng số RWP cho MATLAB...")
W1_rwp = model_rwp.fc1.weight.detach().cpu().numpy()
b1_rwp = model_rwp.fc1.bias.detach().cpu().numpy()
W2_rwp = model_rwp.fc2.weight.detach().cpu().numpy()
b2_rwp = model_rwp.fc2.bias.detach().cpu().numpy()

import scipy.io as sio
sio.savemat("rwp_weights-0.09.mat", {
    "W1": W1_rwp, 
    "b1": b1_rwp, 
    "W2": W2_rwp, 
    "b2": b2_rwp
})

print("Đã lưu trọng số RWP vào file: rwp_weights0.09.mat")

Epoch 1/200: LR=0.2000, TrainLoss=1.2470, TrainAcc=66.74%, TestClean=86.73%, TestNoisy=81.29%
  ↳ New best: Noisy=81.29%, Clean=86.73%
Epoch 2/200: LR=0.1980, TrainLoss=0.4506, TrainAcc=89.16%, TestClean=90.68%, TestNoisy=87.67%
  ↳ New best: Noisy=87.67%, Clean=90.68%
Epoch 3/200: LR=0.1961, TrainLoss=0.3877, TrainAcc=90.59%, TestClean=91.48%, TestNoisy=88.00%
  ↳ New best: Noisy=88.00%, Clean=91.48%
Epoch 4/200: LR=0.1942, TrainLoss=0.3545, TrainAcc=91.32%, TestClean=92.31%, TestNoisy=89.58%
  ↳ New best: Noisy=89.58%, Clean=92.31%
Epoch 5/200: LR=0.1923, TrainLoss=0.3319, TrainAcc=92.00%, TestClean=91.76%, TestNoisy=88.94%
Epoch 6/200: LR=0.1905, TrainLoss=0.3096, TrainAcc=92.47%, TestClean=92.83%, TestNoisy=90.79%
  ↳ New best: Noisy=90.79%, Clean=92.83%
Epoch 7/200: LR=0.1887, TrainLoss=0.2888, TrainAcc=93.07%, TestClean=93.57%, TestNoisy=91.86%
  ↳ New best: Noisy=91.86%, Clean=93.57%
Epoch 8/200: LR=0.1869, TrainLoss=0.2682, TrainAcc=93.58%, TestClean=93.70%, TestNoisy=91.04%
Ep